# 09 — Framework Comparison: RAGAs and DeepEval, Against Your Own Metric

Notebook 08 built `FaithfulnessMetric` yourself: a prompt, a JSON parse, a score. Two real frameworks -- `ragas` and `deepeval` -- ship a faithfulness metric that claims to do the same job. This notebook installs both, runs them on the exact same 20 examples, and compares all three side by side. Not "which framework is best" in the abstract -- which one actually agrees with the judgment calls you already made, on data you've read closely enough to trust your own verdict.


In [ ]:
# pip install ragas deepeval


## A real compatibility snag, found immediately

Neither framework installed cleanly against this project's exact dependency versions -- worth showing rather than hiding, since "does it even install" is itself part of evaluating a framework. `deepeval` 2.5.5 (the version this project resolves to, since the current 4.x line requires a `rich` version older than what the rest of this project already depends on) still imports `from langchain.schema import HumanMessage` -- a path that doesn't exist in the LangChain 1.x installed here. Rather than downgrade LangChain and risk breaking every other notebook in this repo, a small compatibility shim bridges the gap: register a fake `langchain.schema` module that re-exports the classes from where LangChain 1.x actually keeps them, *before* deepeval imports it.


In [ ]:
import sys
import types

import langchain_core.messages as _messages

_shim = types.ModuleType("langchain.schema")
_shim.HumanMessage = _messages.HumanMessage
_shim.AIMessage = _messages.AIMessage
_shim.BaseMessage = _messages.BaseMessage
sys.modules["langchain.schema"] = _shim

import deepeval  # noqa: E402 -- must come after the shim above

print("deepeval imported successfully via compatibility shim")


## Pointing both frameworks at Groq instead of OpenAI

Both frameworks default to calling OpenAI, and this project only has a Groq key. Both support custom model wrappers -- ragas via `LangchainLLMWrapper` around `ChatGroq`, deepeval via a small `DeepEvalBaseLLM` subclass you write yourself.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq

load_dotenv()

MODEL = "openai/gpt-oss-20b"
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# --- ragas wrapper ---
from ragas.llms import LangchainLLMWrapper

ragas_llm = LangchainLLMWrapper(ChatGroq(model=MODEL, temperature=0.0, reasoning_effort="low"))

# --- deepeval wrapper ---
from deepeval.models.base_model import DeepEvalBaseLLM


class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model_name: str = MODEL):
        self.model_name = model_name
        self.client = groq_client
        super().__init__(model_name)

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema=None, _retries: int = 2):
        if schema is not None:
            prompt = prompt + f"\n\nRespond with ONLY valid JSON matching this schema: {schema.model_json_schema()}"
        last_error = None
        for attempt in range(_retries + 1):
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0 if attempt == 0 else 0.3,  # nudge the retry away from repeating the same bad output
                reasoning_effort="low",
            )
            text = response.choices[0].message.content
            if schema is None:
                return text
            try:
                return schema.model_validate_json(text)
            except Exception as e:  # malformed JSON from the model -- real and reproducible on long legal text
                last_error = e
        raise last_error

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name


deepeval_model = GroqModel()
print("Both frameworks wired to Groq.")


Two real snags showed up the moment this ran against the full 20-example set instead of one toy case. First: `deepeval` asks the model for structured JSON, and every so often the JSON comes back malformed and fails to parse -- rerunning the exact same example afterward sometimes succeeds on the first try, which rules out "this specific input always breaks it" and points at something less convenient: hosted LLM inference isn't always bit-for-bit reproducible run to run, even at `temperature=0.0`. `generate()` above retries up to twice for exactly this reason. Second, and bigger: this account's Groq tier has a modest tokens-per-minute cap, and three frameworks times 20 examples fired back-to-back blows straight through it -- covered just below. Both are the same underlying lesson as notebook 12's `max_steps` safety net: any pipeline calling a model needs to plan for occasional, not-fully-predictable failures, not just the ones you can see coming.


## First check: do the two frameworks even agree with each other?

Before touching your own 20 examples, run both on one deliberately unfaithful, made-up test case. If they disagree here, on something this obvious, that's worth knowing before trusting either one on anything harder.


In [ ]:
import asyncio
from ragas.metrics import Faithfulness
from ragas.dataset_schema import SingleTurnSample
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

test_context = "2 plus 2 equals 4. Basic arithmetic."
test_answer = "2+2 equals 5, and also the sky is green."

ragas_metric = Faithfulness(llm=ragas_llm)
ragas_sample = SingleTurnSample(user_input="What is 2+2?", response=test_answer, retrieved_contexts=[test_context])
ragas_score = asyncio.run(ragas_metric.single_turn_ascore(ragas_sample))

deepeval_metric = FaithfulnessMetric(model=deepeval_model, include_reason=True)
deepeval_case = LLMTestCase(input="What is 2+2?", actual_output=test_answer, retrieval_context=[test_context])
deepeval_metric.measure(deepeval_case)

print(f"ragas faithfulness:    {ragas_score:.2f}")
print(f"deepeval faithfulness: {deepeval_metric.score:.2f}  ({deepeval_metric.reason})")


Two claims in that answer: "2+2 equals 5" (directly contradicts the context) and "the sky is green" (unrelated to the context entirely, so unsupported by it either way). If `ragas` scored this `0.0` and `deepeval` scored it `0.5`, that's already a real disagreement on the most obvious possible case -- likely because they're counting differently under the hood (all-or-nothing per response versus per-claim credit). Neither number is "wrong." They're measuring the same idea with different arithmetic, which is exactly the kind of thing "compare frameworks" has to mean in practice, not just "which one is more accurate."


## Running all three on your real 20 examples

Your own `FaithfulnessMetric` from notebook 08, `ragas`, and `deepeval` -- same 20 question/context/answer sets, side by side.


This account's Groq tier caps out at a modest tokens-per-minute budget. Three frameworks times 20 examples, fired back to back with no pause, blows straight through it -- a real `429 Rate limit reached`, not a hypothetical. Two changes handle it: a short pause between examples, and a retry that actually waits out a rate-limit error instead of just trying again immediately (which would just hit the same wall).


In [ ]:
import time


def with_rate_limit_retry(fn, *args, max_attempts: int = 4, **kwargs):
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 15 * (attempt + 1)
                print(f"    rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError(f"Still rate limited after {max_attempts} attempts")


In [ ]:
DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)

FAITHFULNESS_PROMPT = """You are a faithfulness judge. Extract every factual claim in the \
answer below, then mark each as "supported" or "unsupported" based only on the context. \
Return strict JSON only, no other text, in this shape:
{{"claims": [{{"text": "...", "supported": true, "reason": "..."}}]}}

Context:
{context}

Answer:
{answer}"""


def own_faithfulness(context: str, answer: str) -> float:
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": FAITHFULNESS_PROMPT.format(context=context, answer=answer)}],
        temperature=0.0,
        reasoning_effort="low",
    )
    result = json.loads(response.choices[0].message.content)
    claims = result["claims"]
    if not claims:
        return 1.0
    return sum(1 for c in claims if c["supported"]) / len(claims)


def score_one(ex: dict) -> dict:
    own_score = with_rate_limit_retry(own_faithfulness, ex["context_text"], ex["system_answer"])

    r_sample = SingleTurnSample(
        user_input=ex["question"], response=ex["system_answer"], retrieved_contexts=[ex["context_text"]],
    )
    ragas_score = with_rate_limit_retry(lambda: asyncio.run(ragas_metric.single_turn_ascore(r_sample)))

    d_case = LLMTestCase(input=ex["question"], actual_output=ex["system_answer"], retrieval_context=[ex["context_text"]])
    with_rate_limit_retry(deepeval_metric.measure, d_case)

    return {"own": own_score, "ragas": ragas_score, "deepeval": deepeval_metric.score}


results = []
skipped = []
for i, ex in enumerate(examples):
    try:
        scores = score_one(ex)
        results.append({"index": i, **scores})
        time.sleep(3)  # stay under the account's tokens-per-minute budget
    except Exception as e:
        # A real, occasional failure: even at temperature 0, hosted LLM inference isn't always
        # bit-for-bit reproducible, and a judge can produce malformed structured output on a retry
        # just as easily as on a first attempt. Skip and keep going, don't let one bad call sink
        # a 20-example batch -- and don't quietly pretend it didn't happen either.
        skipped.append({"index": i, "error": str(e)[:150]})

print(f"{'#':<4}{'Yours':<8}{'ragas':<8}{'deepeval'}")
for r in results:
    print(f"{r['index']:<4}{r['own']:<8.2f}{r['ragas']:<8.2f}{r['deepeval']:.2f}")

if skipped:
    print(f"\nSkipped {len(skipped)} example(s) after repeated malformed judge output:")
    for s in skipped:
        print(f"  [{s['index']}] {s['error']}")


In [ ]:
avg_own = sum(r["own"] for r in results) / len(results)
avg_ragas = sum(r["ragas"] for r in results) / len(results)
avg_deepeval = sum(r["deepeval"] for r in results) / len(results)

print(f"Average -- yours: {avg_own:.2f}, ragas: {avg_ragas:.2f}, deepeval: {avg_deepeval:.2f}")


## Reading the disagreements, not just the averages

Averages hide exactly the thing worth finding. Pull out every example where the three scores spread out by more than a small margin, and go read those specific answers -- the same way notebook 07 dug into individual disagreements instead of trusting a summary number.


In [ ]:
for r in results:
    scores = [r["own"], r["ragas"], r["deepeval"]]
    spread = max(scores) - min(scores)
    if spread >= 0.4:
        ex = examples[r["index"]]
        print(f"[{r['index']}] spread={spread:.2f} -- yours={r['own']:.2f} ragas={r['ragas']:.2f} deepeval={r['deepeval']:.2f}")
        print(f"     Answer: {ex['system_answer'][:150]}")
        print()


For every example printed above, ask notebook 07's question again: which score matches what you'd conclude reading the answer yourself? That's the actual comparison -- not "which framework has more GitHub stars," but "which one's arithmetic matches a judgment you already trust, on this exact data."


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Custom model wrapper | Code that lets a framework call your own model provider instead of its default |
| Compatibility shim | A small stand-in module that bridges an old import path to where a class actually lives now, without touching the installed package |
| Score spread | How much three different scorers disagree on the identical input -- a signal worth investigating, not averaging away |

Two frameworks, one homemade metric, one dataset -- and real disagreement showed up before you even reached your own 20 examples, on a two-sentence test case. That's not a flaw in either framework. It's the actual state of LLM-based evaluation: different tools encode different judgment calls about what "faithful" means and how partial credit works, and no single number from any of them should be trusted without the same spot-checking you did by hand in notebooks 05 through 07.

**Next up:** notebook `10` builds an evaluation set from scratch, from real source documents, instead of borrowing one -- the skill this whole series has otherwise skipped by using CUAD, SQuAD2, HotpotQA, and MS MARCO.

**Exercises:**

- For the highest-spread example you found, read the full context and answer yourself, then write down which of the three scores you'd actually trust and why.
- Try `ragas`'s `AnswerCorrectness` or `ContextPrecision` metric (both installed already) on a few examples -- does it point at genuinely different information than faithfulness does?
- The `deepeval` compatibility shim only patched three classes (`HumanMessage`, `AIMessage`, `BaseMessage`). If a different deepeval metric needs a fourth class from the old `langchain.schema` path, how would you find out which one, without guessing?
